# Lab 3.3 · Accuracy and wrong but plausible

Day 3 · Session 3 (A scoring harness) · 25 min. Eissa (data governance) will not accept "Daleel is 95%
accurate" from a slide. He wants a harness that executes every reply on the semantic layer, compares it
with the certified answer, separates the wrong answers anyone would notice from the ones nobody would, and a
report he can re-run himself and get the same numbers.

**AI-tutor.** An assistant may help with pandas; the outcome classes and the tolerance are decisions you defend.

**Reference state.** `data/assistant/certified_60.yaml`, the stand-in's recorded replies (synthetic, see
Lab 3.2) and the supplied scorer `harness/score.py`, which you will re-run against your own harness.

**Tasks**
- [ ] 1 · Execute a reply and compare (6')
- [ ] 2 · Classify every outcome (7')
- [ ] 3 · The report a reviewer can re-run (7')
- [ ] 4 · Read the wrong but plausible answers (5')
- [ ] ✱ Stretch · A tolerance for rounding: where does it stop being right?

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-408.git"
    if not Path("/content/ds-408").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-408"], check=True)
    os.chdir("/content/ds-408/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("qamar.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import json
import re
import pandas as pd
import qamar as Q
import qsem as S
import daleel as D

model = S.Model.load("data/semantic/qamar_model.yaml", Q.db(), metrics_dir="data/metrics")
QS = D.load_set("data/assistant/certified_60.yaml")
BYID = {q["id"]: q for q in QS}
TOL = 0.005
for qid in ["C01", "C15", "Q11"]:
    print(qid, BYID[qid]["question"], "\n   reply:", D.ask(BYID[qid], "full", 1).replace("\n", " "))

## 1 · Execute a reply and compare (6')

A reply is text. It may wrap its JSON in a code fence, refuse, or write a sentence instead.

✏️ **Exercise** · Write `run_reply(text, q)`: find the JSON object in the text, return `{"refused": reason}`
for a refusal, `{"value": ...}` from `D.answer(model, query, kind)` for a query, and `{"error": message}`
when there is no JSON or the layer fails. Write `is_right(got, want)`: labels must match exactly, numbers
within a relative tolerance `TOL`.

In [ ]:
def run_reply(text: str, q: dict) -> dict:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


def is_right(got, want, tol: float = TOL) -> bool:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
cases = [("C01", "```json\n{\"metrics\": [\"collection_rate\"], \"by\": [], \"where\": [\"time.month = DATE '2026-03-01'\"]}\n```"),
         ("C01", "{\"metrics\": [\"collection_rate\"], \"where\": [\"time.month = DATE '2026-30-01'\"]}"),
         ("Q09", "The complaint rate was about 36 per 1,000 lines."),
         ("U14", "{\"refuse\": \"restricted\"}")]
try:
    got = [run_reply(t, BYID[i]) for i, t in cases]
except NotImplementedError:
    got = [D.execute(model, D.parse(t), BYID[i]) for i, t in cases]
for (i, t), g in zip(cases, got):
    print(i, "->", g)
try:
    right = is_right(got[0].get("value"), BYID["C01"]["expect"]["value"])
except NotImplementedError:
    right = D.same(got[0].get("value"), BYID["C01"]["expect"]["value"])
Q.check(right and "error" in got[1] and "error" in got[2] and got[3] == {"refused": "restricted"},
        "a fenced reply runs, a bad date and a sentence are errors, a refusal is kept")

## 2 · Classify every outcome (7')

| Outcome | When |
|---|---|
| `right` | the figure is the certified one, within tolerance |
| `wrong_plausible` | a figure between half and twice the certified one, or a valid but wrong group name |
| `wrong_visible` | any other figure |
| `error` | no JSON, the layer failed, or no figure came back |
| `refused_wrongly` | refused an answerable question |
| `refused_right` | refused a must-refuse question |
| `missed_refusal` | answered a must-refuse question |

✏️ **Exercise** · Write `classify(q, out)` and score run 1 of the full configuration: one row per question with
`id`, `lang`, `area`, `difficulty`, `got`, `want`, `outcome`.

In [ ]:
def classify(q: dict, out: dict) -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


def my_score(questions, variant="full", run=1) -> pd.DataFrame:
    rows = []
    for q in questions:
        out = run_reply(D.ask(q, variant, run), q)
        rows.append({"id": q["id"], "lang": q["lang"], "area": q["area"], "difficulty": q["difficulty"],
                     "got": out.get("value", out.get("refused", out.get("error"))),
                     "want": q["expect"].get("value", q["expect"].get("refuse")), "outcome": classify(q, out)})
    return pd.DataFrame(rows)

In [ ]:
try:
    mine = my_score(QS)
except NotImplementedError:
    mine = D.score(model, QS, "full")
supplied = D.score(model, QS, "full")
print(mine.outcome.value_counts().to_string())
Q.check((mine.outcome.values == supplied.outcome.values).all(), "your harness agrees with the supplied scorer on all 60")

🔍 Execution accuracy counts a refusal as right when the question must be refused. Should `missed_refusal`
count with the errors or with the wrong-but-plausible answers? Defend one in a sentence.

## 3 · The report a reviewer can re-run (7')

A figure in a report is only as good as the command that reproduces it. The supplied scorer prints a hash of
every input: the question set, the replies, the model and the metric files.

✏️ **Exercise** · Write `report(table, folder)`: save `results.csv` and a `report.md` with the accuracy, the
wrong-but-plausible rate, a count per outcome, accuracy by area and by language, the input hashes from
`harness/score.py`, and the exact command that re-runs it. Return the accuracy.

In [ ]:
sys.path.insert(0, "harness")
import score as H  # the supplied scorer, harness/score.py

CMD = "uv run python harness/score.py --variant full --runs 1"


def report(table: pd.DataFrame, folder: Path) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
out_dir = Q.lab_dir("D3_3")
try:
    acc = report(mine, out_dir)
except NotImplementedError:
    acc = Q.reference("d3_3_accuracy")
print((out_dir / "report.md").read_text(encoding="utf-8") if (out_dir / "report.md").exists() else "no report written")
rerun = subprocess.run([sys.executable, "harness/score.py", "--variant", "full", "--runs", "1"], capture_output=True, text=True,
                       encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
reviewer = json.loads(rerun.stdout)
print("reviewer's re-run:", reviewer["accuracy"], reviewer["inputs"])
Q.check(abs(reviewer["accuracy"] - acc) < 1e-9, "the reviewer's re-run gives your accuracy")

## 4 · Read the wrong but plausible answers (5')

✏️ **Exercise** · For every `wrong_plausible` or `missed_refusal` row, return the question, the metric and
filters Daleel used, and the certified metric and filters, so a reader sees *what* went wrong.

In [ ]:
def read_plausible(table: pd.DataFrame) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    wp = read_plausible(mine)
except NotImplementedError:
    wp = pd.DataFrame(Q.reference("d3_3_plausible"))
with pd.option_context("display.max_colwidth", 90):
    print(wp.to_string(index=False))
Q.check(set(wp.id) == {"C13", "C15", "Q11"}, "three answers nobody would notice on a slide")

🔍 C13 used the Central region for "Riyadh". Is that a model error, a missing synonym, or a missing field in
the layer? Which fix would you ask Zeyad for, and which test would catch it next time?

## ✱ Stretch · A tolerance for rounding: where does it stop being right?

Score run 1 again with tolerances from 0.1% to 10%. At which tolerance does a wrong collection rate start to
count as right, and what is the largest tolerance you would sign?

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ Stretch not attempted")

## Checkpoint

1. Why execute the reply instead of comparing its SQL or JSON with the certified query?

<details><summary>Answer</summary>Different queries can return the same figure (filters in another order, a
fiscal year written as a date range); the figure is what the manager sees, so the figure is what is scored.</details>

2. Eissa re-runs the command and gets a different accuracy. Name the first thing to compare.

<details><summary>Answer</summary>The input hashes: a different question set, recording, model or metric file
explains a different number before any code does.</details>

**Carry forward.** `artifacts/D3_3/report.md` and `results.csv`. Lab 3.4 starts from the supplied files.